# 03 — Feature Engineering

**Goal:** encode categoricals (`protocol_type`, `service`, `flag`) with `OneHotEncoder`; transform numeric features with `log1p` → `RobustScaler`. Output → `data/feature_matrix.pkl` + fitted encoders in `artifacts/`.

In [9]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

df = pd.read_csv(REPO_ROOT / "data" / "kdd_clean.csv")
print("clean shape:", df.shape)
df.head()

clean shape: (147907, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


## 1. Categorical encoding — OneHotEncoder

In [10]:
from src.preprocessing import CATEGORICAL_COLS, CONTINUOUS_COLS, BINARY_COLS

print("categorical:", CATEGORICAL_COLS)
print("continuous  :", CONTINUOUS_COLS)
print("binary      :", BINARY_COLS)

categorical: ['protocol_type', 'service', 'flag']
continuous  : ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']
binary      : ['land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login']


## 2. Numeric transforms — log1p → RobustScaler

In [3]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# --- categorical ---
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(df[CATEGORICAL_COLS])
cat_names = encoder.get_feature_names_out(CATEGORICAL_COLS)
cat_df = pd.DataFrame(encoded, columns=cat_names, index=df.index)

# --- numeric ---
num_cols = CONTINUOUS_COLS + BINARY_COLS
log_df = np.log1p(df[num_cols].astype(np.float64))
scaler = RobustScaler()
scaled = scaler.fit_transform(log_df)
num_df = pd.DataFrame(scaled, columns=num_cols, index=df.index)

feature_matrix = pd.concat([num_df, cat_df], axis=1)
print("feature_matrix:", feature_matrix.shape)
feature_matrix.head()

feature_matrix: (147907, 122)


,duration,src_bytes,dst_bytes,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0.0,0.424743,0.000000,-0.399367,-0.547411,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.210216,0.000000,0.000000,-0.773706,0.000000,0.000000,0.0,0.0,-1.030693,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,-0.675993,0.000000,0.565492,-0.074525,1.054184,1.000000,0.0,0.0,-1.077814,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.292011,1.415022,-0.219666,-0.160558,0.277287,0.263034,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.264891,0.949390,0.206089,0.790880,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [4]:
# ============================================================
# 3. Build + persist via src (single canonical path)
# ============================================================
from src.encoding import build_feature_matrix

feature_matrix, encoder, scaler = build_feature_matrix(
    df,
    categorical_cols=CATEGORICAL_COLS,
    numeric_cols=num_cols,
    artifact_dir=REPO_ROOT / "artifacts",
    data_dir=REPO_ROOT / "data",
)

assert feature_matrix.shape[1] == len(num_cols) + len(cat_names), "shape mismatch"
print("saved feature_matrix.pkl + artifacts (encoder.pkl, robust_scaler.pkl, feature_names.json)")

[encode] feature_matrix: (147907, 122) (38 numeric + 84 one-hot)
[save] D:\ChinarQAI\task6\v2\QWGAN_IDS\data\feature_matrix.pkl
[save] artifacts/encoder.pkl, robust_scaler.pkl, feature_names.json
saved feature_matrix.pkl + artifacts (encoder.pkl, robust_scaler.pkl, feature_names.json)


## 4. Sanity check — inverse numeric transform

In [5]:
from src.encoding import inverse_numeric

recovered = inverse_numeric(feature_matrix, scaler, num_cols)
orig_log = np.log1p(df[num_cols].astype(np.float64))

err = np.abs(recovered.values - np.expm1(orig_log).values).max()
print(f"max |decoded - original| (after inverse log1p) = {err:.6e}")
assert err < 1e-5, "inverse numeric transform failed"

max |decoded - original| (after inverse log1p) = 2.503395e-06
